In [1]:
# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import random
import numpy as np
import pandas as pd


In [2]:
data = pd.read_csv("breast-cancer.csv")
data

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


Applying Hybrid model(Random Forest and Gradient Booster) without Hyperparamater tuning and Genetic algorithm

In [3]:
data.diagnosis=[1 if each=="M" else 0 for each in data.diagnosis] # Binary classification
y=data.diagnosis.values
x_data=data.drop(["diagnosis","id"],axis=1)  # Replace 'target' with the actual target column name


# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x_data, y, test_size=0.2, random_state=42)

# Train Random Forest and Gradient Boosting models
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)

rf_clf.fit(X_train, y_train)
gb_clf.fit(X_train, y_train)

# Get predictions from both models
rf_preds = rf_clf.predict_proba(X_test)[:, 1]
gb_preds = gb_clf.predict_proba(X_test)[:, 1]

# Combine predictions as features for the meta-classifier
stacked_features = np.column_stack((rf_preds, gb_preds))

# Train a meta-classifier (e.g., Logistic Regression)
meta_clf = LogisticRegression()
meta_clf.fit(stacked_features, y_test)

# Make final predictions
final_preds = meta_clf.predict(stacked_features)

# Evaluate the combined model
accuracy = accuracy_score(y_test, final_preds) * 100
precision = precision_score(y_test, final_preds, average='binary')
recall = recall_score(y_test, final_preds, average='binary')
f1 = f1_score(y_test, final_preds, average='binary')

print("Accuracy for Combined Model (Random Forest + Gradient Boosting):", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy for Combined Model (Random Forest + Gradient Boosting): 95.6140350877193
Precision: 0.9523809523809523
Recall: 0.9302325581395349
F1 Score: 0.9411764705882352


Applying Hybrid model(Random Forest and Gradient Booster) with Hyperparamater tuning and Genetic algorithem

In [4]:
data = pd.read_csv("breast-cancer.csv")
data

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [5]:
data.diagnosis = [1 if each == "M" else 0 for each in data.diagnosis]  # Binary classification
y = data.diagnosis.values
x_data = data.drop(["diagnosis", "id"], axis=1)

In [6]:
# Normalize the data
x_data = (x_data - np.min(x_data)) / (np.max(x_data) - np.min(x_data))

In [7]:
# Define Genetic Algorithm for feature selection
class GeneticAlgorithm:
    def __init__(self, num_features, population_size, num_generations, mutation_rate):
        self.num_features = num_features
        self.population_size = population_size
        self.num_generations = num_generations
        self.mutation_rate = mutation_rate

    def initialize_population(self):
        return [np.random.choice([0, 1], size=self.num_features) for _ in range(self.population_size)]

    def fitness(self, individual, X, y):
        selected_features = np.where(individual == 1)[0]
        if len(selected_features) == 0:
            return 0
        X_selected = X[:, selected_features]
        model = RandomForestClassifier(random_state=42)
        scores = cross_val_score(model, X_selected, y, cv=5, scoring='accuracy')
        return scores.mean()

    def select_parents(self, population, fitness_scores):
        probabilities = fitness_scores / np.sum(fitness_scores)
        parents = random.choices(population, weights=probabilities, k=2)
        return parents

    def crossover(self, parent1, parent2):
        point = random.randint(1, self.num_features - 1)
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        return child1, child2

    def mutate(self, individual):
        for i in range(len(individual)):
            if random.random() < self.mutation_rate:
                individual[i] = 1 - individual[i]
        return individual

    def run(self, X, y):
        population = self.initialize_population()
        for generation in range(self.num_generations):
            fitness_scores = np.array([self.fitness(ind, X, y) for ind in population])
            new_population = []
            for _ in range(self.population_size // 2):
                parent1, parent2 = self.select_parents(population, fitness_scores)
                child1, child2 = self.crossover(parent1, parent2)
                new_population.append(self.mutate(child1))
                new_population.append(self.mutate(child2))
            population = new_population
        best_individual = max(population, key=lambda ind: self.fitness(ind, X, y))
        return np.where(best_individual == 1)[0]

In [8]:

# Run Genetic Algorithm for feature selection
ga = GeneticAlgorithm(num_features=x_data.shape[1], population_size=20, num_generations=50, mutation_rate=0.1)
selected_features = ga.run(x_data.values, y)

In [9]:
# Train Random Forest and Gradient Boosting on selected features
X_selected = x_data.iloc[:, selected_features]
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)

rf_clf.fit(X_train, y_train)
gb_clf.fit(X_train, y_train)


GradientBoostingClassifier(random_state=42)

In [10]:
# Combine predictions using Logistic Regression
rf_preds = rf_clf.predict_proba(X_test)[:, 1]
gb_preds = gb_clf.predict_proba(X_test)[:, 1]
stacked_features = np.column_stack((rf_preds, gb_preds))

meta_clf = LogisticRegression()
meta_clf.fit(stacked_features, y_test)
final_preds = meta_clf.predict(stacked_features)

In [11]:
# Evaluate the combined model
accuracy = accuracy_score(y_test, final_preds) * 100
precision = precision_score(y_test, final_preds)
recall = recall_score(y_test, final_preds)
f1 = f1_score(y_test, final_preds)

print("Accuracy for Combined Model (Random Forest + Gradient Boosting) with Genetic Algorithm:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy for Combined Model (Random Forest + Gradient Boosting) with Genetic Algorithm: 97.36842105263158
Precision: 1.0
Recall: 0.9302325581395349
F1 Score: 0.963855421686747
